# Passive stiffness sweep — F vs d

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from glob import glob

sys.path.insert(0, os.path.join('../../'))
from plot_config import draw_radar, COLORS, FIG_W_SINGLE, FIG_W_DOUBLE, set_font_size
FIG_W = 7.41   # inches

FIG_H = FIG_W * (5.15 / 7.41)   # H/W = 5.15/7.41

GRID = False
FONT_SIZE = 18  # pt — change to rescale all text uniformly
set_font_size(FONT_SIZE)

# Shared axes box (figure fraction) so the two F-vs-d plots match in size;
# the range plot puts its colorbar in the reserved right margin, leaving the
# data axes identical to the 3-case plot. Save full-figure (no tight crop).
AX_RECT   = [0.15, 0.17, 0.68, 0.76]
CBAR_RECT = [0.835, 0.17, 0.025, 0.76]   # gap to axes cut to 25% (0.02 -> 0.005)
plt.rcParams['savefig.bbox'] = None

OUTPUT_DIR = os.path.join('outputs', 'stiffness_sweep')
RANGE_DIR  = os.path.join('outputs', 'stiffness_range')
COLORS     = plt.rcParams['axes.prop_cycle'].by_key()['color']


def available_sweep_k():
    ks = []
    for folder in glob(os.path.join(OUTPUT_DIR, 'K_*')):
        name = os.path.basename(folder)
        try:
            ks.append(float(name.split('_', 1)[1]))
        except ValueError:
            pass
    return sorted(ks)


def load_runs(k):
    pattern = os.path.join(OUTPUT_DIR, f'K_{k:.2f}', f'K_{k:.2f}_run_*.csv')
    return [pd.read_csv(f) for f in sorted(glob(pattern))]


def interp_disp_phase(dfs, phase, col='Force_N', n_pts=500):
    """Interpolate one phase (descent/ascent) onto a common displacement grid."""
    disp_force_pairs = []

    for df in dfs:
        if 'Phase' in df.columns:
            phase_df = df[df['Phase'] == phase].copy()
        else:
            phase_df = df.copy()

        if phase_df.empty:
            continue

        phase_df = phase_df.sort_values('UR5_displacement_m')
        grouped = phase_df.groupby('UR5_displacement_m', as_index=False)[col].mean()

        d_vals = grouped['UR5_displacement_m'].to_numpy()
        f_vals = grouped[col].to_numpy()

        if len(d_vals) < 2:
            continue

        disp_force_pairs.append((d_vals, f_vals))

    if not disp_force_pairs:
        return None, None

    d_max = min(0.02, min(d_vals.max() for d_vals, _ in disp_force_pairs))
    d = np.linspace(0, d_max, n_pts)
    vals = np.array([np.interp(d, d_vals, f_vals) for d_vals, f_vals in disp_force_pairs])
    return d, vals


def available_range_k():
    ks = set()
    for csv_path in glob(os.path.join(RANGE_DIR, 'K_*.csv')):
        stem = os.path.splitext(os.path.basename(csv_path))[0]
        try:
            value_part = stem.split('_', 1)[1]
            value_part = value_part.split('_run_', 1)[0]
            ks.add(float(value_part))
        except ValueError:
            pass
    return sorted(ks)


def load_range_runs(k):
    run_pattern = os.path.join(RANGE_DIR, f'K_{k:.2f}_run_*.csv')
    run_files = sorted(glob(run_pattern))
    if run_files:
        return [pd.read_csv(f) for f in run_files]

    single_path = os.path.join(RANGE_DIR, f'K_{k:.2f}.csv')
    if os.path.exists(single_path):
        return [pd.read_csv(single_path)]

    return []


K_SWEEP = available_sweep_k()
if not K_SWEEP:
    print('No sweep data found in outputs/stiffness_sweep')

data = {k: load_runs(k) for k in K_SWEEP}
for k, runs in data.items():
    print(f'K={k:.2f}  ->  {len(runs)} run(s)')

K_RANGE = available_range_k()
range_data = {k: load_range_runs(k) for k in K_RANGE}
for k, runs in range_data.items():
    print(f'Range K={k:.2f}  ->  {len(runs)} run(s)')
print(f'Range files found: {len(K_RANGE)}')

K=0.10  ->  5 run(s)
K=0.20  ->  5 run(s)
K=0.60  ->  5 run(s)
Range K=0.02  ->  5 run(s)
Range K=0.03  ->  5 run(s)
Range K=0.05  ->  5 run(s)
Range K=0.08  ->  5 run(s)
Range K=0.13  ->  5 run(s)
Range K=0.20  ->  5 run(s)
Range K=0.28  ->  5 run(s)
Range K=0.37  ->  5 run(s)
Range K=0.48  ->  5 run(s)
Range K=0.60  ->  5 run(s)
Range files found: 10


## F vs d

In [2]:
fig = plt.figure(figsize=(FIG_W, FIG_H))
ax  = fig.add_axes(AX_RECT)   # fixed box -> matches the range plot exactly

for idx, k in enumerate(K_SWEEP):
    runs = data[k]
    if not runs:
        continue

    color = COLORS[idx % len(COLORS)]

    phase_specs = [
        ('descent', '-', 0.13, 0.16),
        ('ascent',  '--', 0.10, 0.10),
    ]

    for phase, linestyle, raw_alpha, band_alpha in phase_specs:
        d, vals = interp_disp_phase(runs, phase=phase)
        if vals is None:
            continue

        d_mm = d * 1e3
        mean, std = vals.mean(0), vals.std(0)

        for v in vals:
            ax.plot(d_mm, v, color=color, alpha=raw_alpha, linewidth=1, linestyle=linestyle)

        label = rf'$K_d = {k:.2f}$ N$\cdot$m/rad' if phase == 'descent' else None
        ax.plot(d_mm, mean, color=color, linewidth=2.5, linestyle=linestyle, label=label)
        ax.fill_between(d_mm, mean - std, mean + std, color=color, alpha=band_alpha)

ax.set_xlabel('Distance (mm)')
ax.set_ylabel('Normal force (N)')
ax.legend(loc='upper left')
ax.set_xlim(0, 20)
ax.set_ylim(bottom=0)
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'F_vs_d.pdf'))
plt.show()


## F vs d — passive range

In [3]:
if not K_RANGE:
    print('No range data found in outputs/stiffness_range')
else:
    # Plot 1: mean F vs d across the passive range (same axes box as F_vs_d)
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    ax  = fig.add_axes(AX_RECT)
    cmap = plt.cm.viridis   # perceptually uniform, Nature-friendly stiffness gradient

    for idx, k in enumerate(K_RANGE):
        runs = range_data.get(k, [])
        if not runs:
            continue

        color = cmap(idx / max(1, len(K_RANGE) - 1))
        phase_specs = [
            ('descent', '-', 1.6, 0.95),
            ('ascent',  '--', 1.2, 0.25),
        ]

        for phase, linestyle, lw, alpha in phase_specs:
            d, vals = interp_disp_phase(runs, phase=phase)
            if vals is None:
                continue
            d_mm = d * 1e3
            mean = vals.mean(0)
            ax.plot(d_mm, mean, color=color, linewidth=lw, alpha=alpha, linestyle=linestyle)

    ax.set_xlabel('Distance (mm)')
    ax.set_ylabel('Normal force (N)')
    ax.set_xlim(0, 20)
    ax.set_ylim(bottom=0)

    sm = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=min(K_RANGE), vmax=max(K_RANGE)))
    sm.set_array([])
    cax  = fig.add_axes(CBAR_RECT)   # colorbar in the margin -> data axes stay matched
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(r'$K_d$ (N$\cdot$m/rad)')

    os.makedirs(RANGE_DIR, exist_ok=True)
    fig.savefig(os.path.join(RANGE_DIR, 'F_vs_d_range.pdf'))
    plt.show()

    # Plot 2: effective stiffness (dF/dd) vs Kd with std bars across runs
    kd_vals = []
    k_eff_mean = []
    k_eff_std = []

    for k in K_RANGE:
        runs = range_data.get(k, [])
        if not runs:
            continue

        k_eff_runs = []
        for df in runs:
            if df is None or df.empty:
                continue

            if 'Phase' in df.columns and (df['Phase'] == 'descent').any():
                phase_df = df[df['Phase'] == 'descent'].copy()
            else:
                phase_df = df.copy()

            grouped = (
                phase_df.groupby('UR5_displacement_m', as_index=False)['Force_N']
                        .mean()
                        .sort_values('UR5_displacement_m')
            )

            d_m = grouped['UR5_displacement_m'].to_numpy()
            f_n = grouped['Force_N'].to_numpy()

            if len(d_m) < 2:
                continue

            delta_d = d_m[-1] - d_m[0]
            if delta_d <= 0:
                continue

            delta_f = f_n[-1] - f_n[0]
            k_eff_runs.append(delta_f / delta_d)

        if k_eff_runs:
            kd_vals.append(k)
            k_eff_mean.append(float(np.mean(k_eff_runs)))
            if len(k_eff_runs) > 1:
                k_eff_std.append(float(np.std(k_eff_runs, ddof=1)))
            else:
                k_eff_std.append(0.0)

    if not kd_vals:
        print('No valid range data to compute stiffness')
    else:
        kd_vals = np.array(kd_vals)
        k_eff_mean = np.array(k_eff_mean)
        k_eff_std = np.array(k_eff_std)

        sort_idx = np.argsort(kd_vals)
        x = kd_vals[sort_idx]
        y = k_eff_mean[sort_idx]
        yerr = k_eff_std[sort_idx]

        STIFF_COLOR = COLORS[3]   # NPG navy - distinct from the normal-force palette
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.errorbar(
            x, y, yerr=yerr, fmt='o', linestyle='--', color=STIFF_COLOR,
            ecolor=STIFF_COLOR, elinewidth=1.2, capsize=4, capthick=1.2, alpha=0.95
        )
        ax.set_xlabel(r'$K_d$ (N$\cdot$m/rad)')
        ax.set_ylabel(r'$\Delta F/\Delta d$ (N/m)')
        ax.set_xlim(0, None)
        ax.set_ylim(bottom=0)
        plt.tight_layout()
        fig.savefig(os.path.join(RANGE_DIR, 'Keff_vs_Kd_range.pdf'))
        plt.show()
